# Faster, more accurate threat detection and response  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 1.5.d: Compute SOC incident-response metrics

**Skill:** Detect Attacks, Collaborate · **Maps to:** 1.5.B · **Language:** Python 3 (pytest)

## Problem
A SOC dashboard needs two key metrics to measure how well AI-assisted response is performing:

1. **MTTR** (Mean Time to Respond): the average seconds from detection to resolution,
   computed across all incidents in the dataset.
2. **auto_resolved**: the count of incidents that were closed automatically by an AI playbook
   (no human analyst required).

Implement `solve(incidents)` that reads a list of incident records and returns both metrics.

## Function contract
Implement `solve(incidents: list[dict]) -> dict` in the code editor
(see the Reference section below).

## Input
`incidents` -- a list of dicts, each with:
- `id` (str): unique incident identifier
- `detected_at` (int): Unix timestamp (seconds) when detected
- `resolved_at` (int): Unix timestamp (seconds) when resolved; always > detected_at
- `severity` (str): one of `"critical"`, `"high"`, `"medium"`, `"low"`
- `auto` (bool): True if auto-resolved, False if human-resolved

## Output
A dict with exactly two keys:
- `"mttr"` (float): mean response time in seconds across all incidents
- `"auto_resolved"` (int): count of incidents where `auto == True`

Formula:
  `mttr = sum(resolved_at - detected_at for each incident) / len(incidents)`

Return `{"mttr": 0.0, "auto_resolved": 0}` for empty input.

## Examples
```python
incidents = [
    {"id": "INC-001", "detected_at": 1000, "resolved_at": 1120, "severity": "critical", "auto": False},
    {"id": "INC-002", "detected_at": 2000, "resolved_at": 2080, "severity": "high",     "auto": True},
]
# response times: 120s, 80s -> mean = 100.0; auto_resolved = 1
solve(incidents) -> {"mttr": 100.0, "auto_resolved": 1}

solve([]) -> {"mttr": 0.0, "auto_resolved": 0}

incidents = [
    {"id": "INC-003", "detected_at": 0, "resolved_at": 300, "severity": "low", "auto": True},
]
solve(incidents) -> {"mttr": 300.0, "auto_resolved": 1}
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(incidents: list[dict]) -> dict
```

### Input (`incidents`)
A list of incident dicts. Each dict has exactly these keys:

| Key           | Type  | Description                                                          |
|---------------|-------|----------------------------------------------------------------------|
| `id`          | str   | Unique incident identifier string (e.g. `"INC-001"`)                |
| `detected_at` | int   | Unix timestamp (seconds) when the incident was detected              |
| `resolved_at` | int   | Unix timestamp (seconds) when the incident was resolved; always > detected_at |
| `severity`    | str   | One of `"critical"`, `"high"`, `"medium"`, `"low"`                  |
| `auto`        | bool  | `True` if the incident was resolved automatically; `False` if human-resolved |

Example single entry:
```python
{"id": "INC-001", "detected_at": 1000, "resolved_at": 1120, "severity": "critical", "auto": False}
```

### Output (`result`)
A dict with exactly two keys:

| Key            | Type  | Description                                                           |
|----------------|-------|-----------------------------------------------------------------------|
| `"mttr"`       | float | Mean time to respond in seconds, averaged across ALL incidents.       |
| `"auto_resolved"` | int | Count of incidents where `auto == True`.                           |

`mttr` is computed as:
  `sum(resolved_at - detected_at for each incident) / len(incidents)`

If `incidents` is empty, return `{"mttr": 0.0, "auto_resolved": 0}`.

The `mttr` value is a Python `float` (may be a whole number like `120.0`).
The `auto_resolved` value is a Python `int`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- `incidents` is a list (may be empty).
- `resolved_at > detected_at` always; all timestamps are non-negative integers.
- `severity` is always one of the four valid strings; implementations need not validate it.
- No incident has a negative response time.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 1.5.d. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests).

solve(incidents) -> {"mttr": float, "auto_resolved": int}
  mttr = mean(resolved_at - detected_at) across all incidents.
  auto_resolved = count of incidents with auto == True.
  Empty input returns {"mttr": 0.0, "auto_resolved": 0}.
"""
from typing import List, Dict, Any


def solve(incidents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Compute SOC MTTR and auto-resolved count from a list of incident records."""
    if not incidents:
        return {"mttr": 0.0, "auto_resolved": 0}

    total_time = sum(inc["resolved_at"] - inc["detected_at"] for inc in incidents)
    auto_count = sum(1 for inc in incidents if inc.get("auto") is True)

    return {
        "mttr": total_time / len(incidents),
        "auto_resolved": auto_count,
    }


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _inc(id_, detected, resolved, severity="high", auto=False):
    return {"id": id_, "detected_at": detected, "resolved_at": resolved,
            "severity": severity, "auto": auto}


# ---- visible cases -------------------------------------------------------

def test_empty_input():
    result = impl.solve([])
    assert result == {"mttr": 0.0, "auto_resolved": 0}


def test_single_incident_manual():
    result = impl.solve([_inc("INC-001", 1000, 1120, "critical", False)])
    assert result["mttr"] == 120.0
    assert result["auto_resolved"] == 0


def test_single_incident_auto():
    result = impl.solve([_inc("INC-002", 2000, 2060, "low", True)])
    assert result["mttr"] == 60.0
    assert result["auto_resolved"] == 1


def test_two_incidents_mean():
    incidents = [
        _inc("INC-003", 1000, 1120, "critical", False),  # 120s
        _inc("INC-004", 2000, 2080, "high", True),       # 80s
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 100.0
    assert result["auto_resolved"] == 1


def test_all_auto_resolved():
    incidents = [
        _inc("INC-005", 0, 30, "low", True),
        _inc("INC-006", 0, 60, "medium", True),
        _inc("INC-007", 0, 90, "high", True),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 60.0
    assert result["auto_resolved"] == 3


# ---- hidden / edge cases -------------------------------------------------

def test_no_auto_resolved():
    incidents = [
        _inc("INC-010", 100, 700, "critical", False),  # 600s
        _inc("INC-011", 200, 500, "high", False),       # 300s
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 450.0
    assert result["auto_resolved"] == 0


def test_mttr_is_float():
    result = impl.solve([_inc("INC-020", 0, 100, "medium", False)])
    assert isinstance(result["mttr"], float)


def test_auto_resolved_is_int():
    result = impl.solve([_inc("INC-021", 0, 50, "low", True)])
    assert isinstance(result["auto_resolved"], int)


def test_result_has_exactly_two_keys():
    result = impl.solve([_inc("INC-030", 0, 200)])
    assert set(result.keys()) == {"mttr", "auto_resolved"}


def test_non_integer_mean():
    # 3 incidents with response times 10, 20, 30 -> mean = 20.0 (exact)
    incidents = [
        _inc("A", 0, 10),
        _inc("B", 0, 20),
        _inc("C", 0, 30),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 20.0


def test_fractional_mean():
    # 2 incidents: 10s and 11s -> mean = 10.5
    incidents = [
        _inc("X", 1000, 1010),
        _inc("Y", 2000, 2011),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 10.5


def test_large_dataset():
    # 100 incidents, all 300s response time, 50 auto -> mttr=300.0, auto=50
    incidents = [_inc(f"INC-{i}", i * 1000, i * 1000 + 300, "high", i % 2 == 0)
                 for i in range(100)]
    result = impl.solve(incidents)
    assert result["mttr"] == 300.0
    assert result["auto_resolved"] == 50


def test_mixed_severities_all_auto():
    incidents = [
        _inc("M1", 0, 45, "low", True),
        _inc("M2", 0, 45, "medium", True),
        _inc("M3", 0, 45, "high", True),
        _inc("M4", 0, 45, "critical", False),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 45.0
    assert result["auto_resolved"] == 3


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
